# ArduMedics Notebook 01: YOLOv8n-Pose Fall Detection Training

**Project**: ArduMedics - AI-Powered Healthcare Robot  
**Component**: Feature 4 - Camera-Based Fall Detection  
**Model**: YOLOv8n-Pose (Nano - Optimized for Raspberry Pi 5 Edge Deployment)  
**Author**: ArduMedics AI Team  
**Notebook**: 01 / 06  
**Previous Notebook**: None (First in series)  
**Next Notebook**: 02_YOLOv8s_Pose_Fall_Detection_Training  

---

## Objective
Train the **lightweight YOLOv8n-Pose** model on ALL available fall detection + pose estimation
datasets for maximum accuracy while keeping the model small enough for Raspberry Pi 5 deployment.
This is our **primary deployment model** (fastest inference on edge).

## Training Strategy
- **Model Size**: Nano (3.2M params) - Best for Pi 5 real-time inference
- **Epochs**: 150 (with patience=40 early stopping)
- **Optimizer**: AdamW with cosine annealing LR
- **Image Size**: 640x640
- **Augmentation**: Mosaic + MixUp + standard augmentations
- **Export Formats**: NCNN (primary for Pi 5), TFLite (INT8), ONNX (universal)

## Datasets Used
### Roboflow Pose Datasets (Keypoint-annotated for YOLOv8-Pose training):
1. **Falling Pose Estimation** (635 images) - https://universe.roboflow.com/humna-pose-data/falling-pose-estimation
2. **YOLOv8-Pose Fall Detection** (474 images, 2-class: fall/not-fallen) - https://universe.roboflow.com/yolo-xvnzo/yolov8-pose-utovc

> **NOTE**: Dataset `nafzzan/falling-pose-estimation-0xme8` was removed as it is a **verified duplicate** of
> Dataset 1 (same 635 images, pixel-identical). Only differences: different Roboflow filename hashes and
> class name ("Falling" vs "person"). Using both would cause data leakage.
### Kaggle Fall Detection Datasets (For additional frame extraction + evaluation):
4. **UR Fall Detection Dataset** - https://www.kaggle.com/datasets/shahliza27/ur-fall-detection-dataset
5. **Fall Detection Dataset (Images)** - https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset
6. **Le2i Fall Dataset** - https://www.kaggle.com/datasets/tuyenldvn/falldataset-imvia
7. **Multiple Cameras Fall Dataset** - https://www.kaggle.com/datasets/soumicksarker/multiple-cameras-fall-dataset
8. **Fall Video Dataset (Combined)** - https://www.kaggle.com/datasets/payutch/fall-video-dataset

**Estimated Training Time**: ~8-10 hours on T4 GPU (well within 12-hour Kaggle Save Version limit)


---
## Step 1: Environment Setup

Install all required packages. This notebook uses:
- `ultralytics` for YOLOv8-Pose training and export
- `roboflow` for downloading pose-annotated datasets
- `opencv-python` for video frame extraction
- `sahi` for Slicing Aided Hyper Inference (optional boost)

In [1]:
# ============================================================
# OUTPUT MANAGEMENT UTILITIES (ArduMedics Standard)
# Suppresses noisy output while keeping important logs
# ============================================================

import os, sys, warnings, contextlib, io

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

os.environ['OPENCV_LOG_LEVEL'] = 'ERROR'
os.environ['OPENCV_VIDEOIO_DEBUG'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['FLAGS_logtostderr'] = '0'
os.environ['GLOG_minloglevel'] = '3'
os.environ['YOLO_AUTODOWNLOAD'] = '1'

class suppress_output:
    def __enter__(self):
        self._orig = sys.stdout
        sys.stdout = io.StringIO()
        return self
    def __exit__(self, *a):
        sys.stdout = self._orig
        return False

_real_stdout = sys.stdout
def important_print(msg, end='\n'):
    _real_stdout.write(str(msg) + end)
    _real_stdout.flush()

print('[ArduMedics] Output management loaded')


[ArduMedics] Output management loaded


In [2]:
# ============================================================
# STEP 1: Install required packages
# Notebook: 01 | Step: 1 of 10
# After this: Download datasets from Roboflow + Kaggle
# ============================================================

!pip install -qq ultralytics roboflow opencv-python-headless sahi

import os
import shutil
import yaml
import json
import cv2
cv2.setLogLevel(0)
import glob
import random
import numpy as np
from pathlib import Path
from datetime import datetime

# Verify ultralytics version (8.1+ required for proper pose export)
from ultralytics import YOLO
print(f"Ultralytics version: {YOLO.__module__}")
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("✓ Step 1 complete: Environment ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dopamine-rl 4.1.2 requires 

---
## Step 2: Load Roboflow Pose Datasets

These datasets have **keypoint annotations** in YOLOv8-Pose format, which is exactly
what we need for training the pose estimation model. We load **2 Roboflow datasets**
(Dataset 3 removed — verified duplicate of Dataset 1) and merge them.

**Upload Instructions**: Upload as a **single ZIP** to Kaggle containing two subfolders
`falling_pose_estimation/` and `yolov8_pose_fall/`. The code will automatically find
datasets regardless of the Kaggle mount path. Do NOT upload extracted folders directly
(exceeds Kaggle's 1000-file limit) and do NOT upload two separate ZIPs (data.yaml conflict).

**Class Mapping Strategy**:
- Dataset 1 (humna): class 0 = `person` → **remap to class 0 = `fall`** (all images show falling poses)
- Dataset 2 (yolo-xvnzo): class 0 = `fall`, class 1 = `not-fallen` → **keep as-is**
- Unified: nc=2, names=['fall', 'not-fallen']

**Fallback**: If no pre-uploaded dataset is found, download via Roboflow SDK
(set `ROBOFLOW_API_KEY` env var or the code will prompt with instructions).


In [3]:
# ============================================================
# STEP 2: Load Roboflow Pose Datasets
# Notebook: 01 | Step: 2 of 10
# After this: Download Kaggle fall datasets for frame extraction
# ============================================================
#
# ROBUST DATASET LOADING (handles ALL Kaggle path variations):
#   1. Recursively scan ALL of /kaggle/input/ for data.yaml files
#   2. Auto-categorize as Dataset 1 or Dataset 2 based on folder name
#   3. Always COPY to /kaggle/working/datasets/ (writable!)
#   4. Never write to /kaggle/input/ (read-only!)
#
# UPLOAD INSTRUCTIONS:
#   - Upload as a SINGLE ZIP containing both subfolders (Kaggle auto-extracts)
#   - Structure inside ZIP: ardumedics-roboflow-pose-datasets.zip
#       ├── falling_pose_estimation/ (data.yaml + train/valid/test)
#       └── yolov8_pose_fall/ (data.yaml + train/valid/test)
#   - Do NOT upload two separate ZIPs (data.yaml conflict)
#   - Do NOT upload extracted folders directly (exceeds 1000-file limit)
#
# CLASS MAPPING:
#   Dataset 1 (humna): class 0='person' → remap to class 0='fall'
#   Dataset 2 (yolo-xvnzo): class 0='fall', class 1='not-fallen' → keep as-is
#   Unified: nc=2, names=['fall', 'not-fallen']

DATASET_DIR = "/kaggle/working/datasets"
os.makedirs(DATASET_DIR, exist_ok=True)

roboflow_datasets_loaded = False

# ── ROBUST: Recursively find ALL data.yaml in /kaggle/input/ ──
# Kaggle mounts datasets at unpredictable paths like:
#   /kaggle/input/ardumedics-roboflow-pose-datasets/
#   /kaggle/input/datasets/username/ardumedics-roboflow-pose-datasets/
#   /kaggle/input/datasets/username/ardumedics-roboflow-pose-datasets/ardumedics-roboflow-pose-datasets/
# We must find data.yaml REGARDLESS of the path depth.

DATASET1_PATTERNS = ['falling_pose_estimation', 'falling-pose-estimation', 
                     'Falling pose estimation', 'humna']
DATASET2_PATTERNS = ['yolov8_pose_fall', 'yolov8-pose', 
                     'yolov8-pose.v1i', 'yolov8-pose.v1xme8', 'yolo-xvnzo']

found_ds1 = None  # Path to Dataset 1 (falling pose)
found_ds2 = None  # Path to Dataset 2 (yolov8 pose)

if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'data.yaml' in files:
            # Categorize based on path name
            root_lower = root.lower()
            if any(p.lower() in root_lower for p in DATASET1_PATTERNS):
                found_ds1 = root
                print(f"  Found Dataset 1 at: {root}")
            elif any(p.lower() in root_lower for p in DATASET2_PATTERNS):
                found_ds2 = root
                print(f"  Found Dataset 2 at: {root}")
            else:
                # Unknown data.yaml — try to auto-assign
                if found_ds1 is None:
                    found_ds1 = root
                    print(f"  Found unknown dataset, assigning as Dataset 1: {root}")
                elif found_ds2 is None:
                    found_ds2 = root
                    print(f"  Found unknown dataset, assigning as Dataset 2: {root}")

# ── Copy found datasets to writable directory ──
if found_ds1:
    dst = f"{DATASET_DIR}/falling_pose_1"
    if not os.path.exists(dst):
        shutil.copytree(found_ds1, dst)
    img_count = 0
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = os.path.join(found_ds1, split, 'images')
        if os.path.exists(img_dir):
            img_count += len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
    print(f"  Dataset 1 → falling_pose_1/ ({img_count} images)")

if found_ds2:
    dst = f"{DATASET_DIR}/falling_pose_2"
    if not os.path.exists(dst):
        shutil.copytree(found_ds2, dst)
    img_count = 0
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = os.path.join(found_ds2, split, 'images')
        if os.path.exists(img_dir):
            img_count += len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
    print(f"  Dataset 2 → falling_pose_2/ ({img_count} images)")

if found_ds1 or found_ds2:
    roboflow_datasets_loaded = True
    print("  Datasets copied to writable directory!")

# ── Fallback: Roboflow SDK download ──
if not roboflow_datasets_loaded:
    print("[Fallback] Downloading from Roboflow SDK...")
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "YOUR_ROBOFLOW_API_KEY_HERE")
    
    if ROBOFLOW_API_KEY == "YOUR_ROBOFLOW_API_KEY_HERE":
        print("=" * 60)
        print("ERROR: No datasets found in /kaggle/input/ and Roboflow API key not set!")
        print("=" * 60)
        print("Either:")
        print("  1. Upload a SINGLE ZIP as 'ardumedics-roboflow-pose-datasets'")
        print("     (ZIP must contain falling_pose_estimation/ + yolov8_pose_fall/)")
        print("  OR")
        print("  2. Set ROBOFLOW_API_KEY env var and re-run")
        print("=" * 60)
        raise ValueError("No datasets available and Roboflow API key not configured")
    
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    # Dataset 1: Falling Pose Estimation (635 images)
    print("Downloading Falling Pose Estimation (635 images)...")
    with suppress_output():
        project1 = rf.workspace("humna-pose-data").project("falling-pose-estimation")
        version1 = project1.version(2)
        dataset1 = version1.download("yolov8", location=f"{DATASET_DIR}/falling_pose_1")
    print("  Done!")
    
    # Dataset 2: YOLOv8-Pose Fall Detection (474 images)
    print("Downloading YOLOv8-Pose Fall Detection (474 images)...")
    with suppress_output():
        project2 = rf.workspace("yolo-xvnzo").project("yolov8-pose-utovc")
        version2 = project2.version(3)
        dataset2 = version2.download("yolov8", location=f"{DATASET_DIR}/falling_pose_2")
    print("  Done!")

print("\nStep 2 complete: Pose datasets ready")

  Found Dataset 1 at: /kaggle/input/datasets/nishatfifa/ardumedics-roboflow-pose-datasets/ardumedics-roboflow-pose-datasets/falling_pose_estimation
  Found Dataset 2 at: /kaggle/input/datasets/nishatfifa/ardumedics-roboflow-pose-datasets/ardumedics-roboflow-pose-datasets/yolov8_pose_fall
  Dataset 1 → falling_pose_1/ (635 images)
  Dataset 2 → falling_pose_2/ (474 images)
  Datasets copied to writable directory!

Step 2 complete: Pose datasets ready


---
## Step 3: Download Kaggle Fall Datasets & Extract Frames

The Kaggle datasets contain fall/non-fall images and videos. We:
1. Add image-based datasets directly to training data (with auto-labeling using pre-trained YOLOv8-Pose)
2. Extract frames from video datasets for evaluation (Notebook 06)

**Kaggle Datasets**: Add these to your Kaggle notebook before running:
- `shahliza27/ur-fall-detection-dataset`
- `uttejkumarkandagatla/fall-detection-dataset`
- `tuyenldvn/falldataset-imvia`
- `soumicksarker/multiple-cameras-fall-dataset`
- `payutch/fall-video-dataset`

In [4]:
# ============================================================
# STEP 3: Download Kaggle fall datasets & extract frames
# Notebook: 01 | Step: 3 of 10
# After this: Merge all datasets into unified YOLOv8-Pose format
# ============================================================

# Kaggle datasets are auto-mounted at /kaggle/input/
# Check what's available
kaggle_input = "/kaggle/input"
print("Available Kaggle datasets:")
for d in sorted(os.listdir(kaggle_input)):
    print(f"  - {d}")

# ---- Frame extraction function for video datasets ----
def extract_frames_from_videos(video_dir, output_dir, sample_rate=5):
    """
    Extract frames from video files at given sample rate.
    
    Args:
        video_dir: Directory containing video files
        output_dir: Directory to save extracted frames
        sample_rate: Extract 1 frame every N frames (default: every 5th frame)
    
    Returns:
        Number of frames extracted
    """
    os.makedirs(output_dir, exist_ok=True)
    count = 0
    
    video_extensions = ['.avi', '.mp4', '.mov', '.mkv', '.wmv']
    video_files = []
    for ext in video_extensions:
        video_files.extend(glob.glob(os.path.join(video_dir, '**', f'*{ext}'), recursive=True))
    
    for vf in video_files:
        cap = cv2.VideoCapture(vf)
        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % sample_rate == 0:
                frame_path = os.path.join(output_dir, f"frame_{count:06d}.jpg")
                cv2.imwrite(frame_path, frame)
                count += 1
            frame_idx += 1
        cap.release()
    
    return count

# ---- Process UR Fall Detection Dataset ----
# URL: https://www.kaggle.com/datasets/shahliza27/ur-fall-detection-dataset
ur_fall_path = None
for d in os.listdir(kaggle_input):
    if 'ur-fall' in d.lower():
        ur_fall_path = os.path.join(kaggle_input, d)
        break

if ur_fall_path:
    print(f"\nProcessing UR Fall Dataset: {ur_fall_path}")
    ur_frames_dir = f"{DATASET_DIR}/ur_fall_frames"
    n_frames = extract_frames_from_videos(ur_fall_path, ur_frames_dir, sample_rate=3)
    print(f"  Extracted {n_frames} frames")

# ---- Process Fall Detection Images Dataset ----
# URL: https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset
fall_img_path = None
for d in os.listdir(kaggle_input):
    if 'fall-detection-dataset' in d.lower():
        fall_img_path = os.path.join(kaggle_input, d)
        break

if fall_img_path:
    print(f"\nProcessing Fall Detection Images: {fall_img_path}")
    # Copy images directly (these are already image files)
    fall_img_dest = f"{DATASET_DIR}/fall_detection_images"
    os.makedirs(fall_img_dest, exist_ok=True)
    img_count = 0
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        for f in glob.glob(os.path.join(fall_img_path, '**', ext), recursive=True):
            shutil.copy2(f, os.path.join(fall_img_dest, f"img_{img_count:06d}.jpg"))
            img_count += 1
    print(f"  Copied {img_count} images")

# ---- Process Le2i Fall Dataset ----
# URL: https://www.kaggle.com/datasets/tuyenldvn/falldataset-imvia
le2i_path = None
for d in os.listdir(kaggle_input):
    if 'falldataset' in d.lower() or 'le2i' in d.lower() or 'imvia' in d.lower():
        le2i_path = os.path.join(kaggle_input, d)
        break

if le2i_path:
    print(f"\nProcessing Le2i Fall Dataset: {le2i_path}")
    le2i_frames_dir = f"{DATASET_DIR}/le2i_frames"
    n_frames = extract_frames_from_videos(le2i_path, le2i_frames_dir, sample_rate=5)
    print(f"  Extracted {n_frames} frames")

# ---- Process Multiple Cameras Fall Dataset ----
# URL: https://www.kaggle.com/datasets/soumicksarker/multiple-cameras-fall-dataset
multi_cam_path = None
for d in os.listdir(kaggle_input):
    if 'multiple-cameras' in d.lower():
        multi_cam_path = os.path.join(kaggle_input, d)
        break

if multi_cam_path:
    print(f"\nProcessing Multiple Cameras Fall Dataset: {multi_cam_path}")
    multi_frames_dir = f"{DATASET_DIR}/multi_cam_frames"
    n_frames = extract_frames_from_videos(multi_cam_path, multi_frames_dir, sample_rate=5)
    print(f"  Extracted {n_frames} frames")

# ---- Process Fall Video Dataset (Combined) ----
# URL: https://www.kaggle.com/datasets/payutch/fall-video-dataset
fall_vid_path = None
for d in os.listdir(kaggle_input):
    if 'fall-video' in d.lower():
        fall_vid_path = os.path.join(kaggle_input, d)
        break

if fall_vid_path:
    print(f"\nProcessing Fall Video Dataset: {fall_vid_path}")
    fallvid_frames_dir = f"{DATASET_DIR}/fall_video_frames"
    n_frames = extract_frames_from_videos(fall_vid_path, fallvid_frames_dir, sample_rate=5)
    print(f"  Extracted {n_frames} frames")

# Disk status check after processing all datasets
disk_stat = shutil.disk_usage("/kaggle/working")
print(f"\nDisk usage: {disk_stat.used / 1e9:.1f} GB / {disk_stat.total / 1e9:.1f} GB ({disk_stat.free / 1e9:.1f} GB free)")
if disk_stat.free < 5e9:
    print("WARNING: Low disk space! Cleaning up temporary files...")
    for zf in Path("/kaggle/working").rglob("*.zip"):
        zf.unlink()
        print(f"  Deleted: {zf}")

print("\n\u2713 Step 3 complete: Kaggle datasets processed")

Available Kaggle datasets:
  - datasets

Disk usage: 0.1 GB / 21.0 GB (20.9 GB free)

✓ Step 3 complete: Kaggle datasets processed


---
## Step 4: Auto-Label Kaggle Frames with Pre-trained YOLOv8-Pose

The Kaggle image/video datasets don't have keypoint annotations. We use the pre-trained
COCO YOLOv8n-Pose model to **auto-label** these images, then add them to our training set.
This is a form of **pseudo-labeling** that significantly expands our training data.

In [5]:
# ============================================================
# STEP 4: Auto-label Kaggle frames using pre-trained YOLOv8-Pose
# Notebook: 01 | Step: 4 of 10
# After this: Merge all datasets into unified training set
# ============================================================

# Load pre-trained COCO pose model for auto-labeling
auto_label_model = YOLO('yolov8n-pose.pt')

def auto_label_images(images_dir, output_labels_dir, conf_threshold=0.5):
    """
    Run pre-trained YOLOv8-Pose on images and save keypoint predictions as labels.
    Only saves labels where a person is detected with confidence > threshold.
    
    Args:
        images_dir: Directory containing images to label
        output_labels_dir: Directory to save YOLOv8-Pose format labels
        conf_threshold: Minimum confidence for keeping predictions
    
    Returns:
        Number of successfully labeled images
    """
    os.makedirs(output_labels_dir, exist_ok=True)
    labeled_count = 0
    
    image_files = []
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        image_files.extend(glob.glob(os.path.join(images_dir, ext)))
    
    print(f"  Auto-labeling {len(image_files)} images from {images_dir}...")
    
    for img_path in image_files:
        results = auto_label_model(img_path, verbose=False)
        
        if len(results) > 0 and results[0].keypoints is not None:
            result = results[0]
            # Get boxes with person class (0) and high confidence
            boxes = result.boxes
            keypoints = result.keypoints
            
            if boxes is not None and len(boxes) > 0:
                # Filter by confidence
                mask = boxes.conf >= conf_threshold
                if mask.any():
                    img_h, img_w = result.orig_shape
                    
                    # Write label file in YOLOv8-Pose format
                    img_name = Path(img_path).stem
                    label_path = os.path.join(output_labels_dir, f"{img_name}.txt")
                    
                    with open(label_path, 'w') as f:
                        for i in range(len(boxes)):
                            if mask[i]:
                                # Class 0 = person
                                cls = int(boxes.cls[i])
                                # Bounding box (normalized)
                                box = boxes.xywhn[i].cpu().numpy()
                                # Keypoints (normalized)
                                kpts = keypoints.xyn[i].cpu().numpy().flatten()
                                
                                # Format: class cx cy w h kp1x kp1y kp1v kp2x kp2y kp2v ...
                                line = f"{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}"
                                for k in range(0, len(kpts), 3):
                                    line += f" {kpts[k]:.6f} {kpts[k+1]:.6f} {int(kpts[k+2])}"
                                f.write(line + "\n")
                    labeled_count += 1
    
    return labeled_count

# Auto-label each extracted frame set
labeled_dirs = {}
for frame_dir_name in ['ur_fall_frames', 'le2i_frames', 'multi_cam_frames', 
                        'fall_video_frames', 'fall_detection_images']:
    frame_dir = os.path.join(DATASET_DIR, frame_dir_name)
    if os.path.exists(frame_dir):
        label_dir = os.path.join(DATASET_DIR, f"{frame_dir_name}_labels")
        n_labeled = auto_label_images(frame_dir, label_dir, conf_threshold=0.5)
        labeled_dirs[frame_dir_name] = {
            'images': frame_dir,
            'labels': label_dir,
            'count': n_labeled
        }
        print(f"  Labeled {n_labeled} images from {frame_dir_name}")

print("\n✓ Step 4 complete: Auto-labeling finished")


✓ Step 4 complete: Auto-labeling finished


---
## Step 5: Merge All Datasets into Unified YOLOv8-Pose Format

Combine the 2 Roboflow pose datasets + auto-labeled Kaggle frames into a single
unified dataset with proper train/val/test split.

In [6]:
# ============================================================
# STEP 5: Merge all datasets into unified YOLOv8-Pose format
# Notebook: 01 | Step: 5 of 10
# After this: Verify dataset integrity and create data.yaml
# ============================================================

UNIFIED_DIR = f"{DATASET_DIR}/ardumedics_unified_pose"

def remap_label_file(src_label, dst_label, class_map):
    """Remap class IDs in a YOLOv8 label file."""
    with open(src_label, 'r') as f:
        lines = f.readlines()
    remapped = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            old_cls = int(parts[0])
            if old_cls in class_map:
                parts[0] = str(class_map[old_cls])
        remapped.append(' '.join(parts))
    with open(dst_label, 'w') as f:
        f.write('\n'.join(remapped) + '\n')

def merge_datasets(roboflow_datasets, auto_labeled_dirs, output_dir, train_ratio=0.8, val_ratio=0.15):
    """
    Merge multiple YOLOv8-Pose datasets into one unified dataset.
    
    Args:
        roboflow_datasets: List of dicts with 'path' and 'class_map' for each Roboflow dataset
        auto_labeled_dirs: Dict of auto-labeled image/label directories
        output_dir: Output directory for unified dataset
        train_ratio: Fraction for training set
        val_ratio: Fraction for validation set (rest goes to test)
    
    Returns:
        Path to unified dataset
    """

    # Create output directories
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    all_pairs = []  # (image_path, label_path, class_map) tuples
    
    # ---- Collect from Roboflow datasets (already split, re-collect and re-split) ----
    for ds_cfg in roboflow_datasets:
        ds_path = str(ds_cfg['path'])
        for split in ['train', 'valid', 'val', 'test']:
            img_dir = os.path.join(ds_path, split, 'images')
            lbl_dir = os.path.join(ds_path, split, 'labels')
            if os.path.exists(img_dir) and os.path.exists(lbl_dir):
                for img_file in os.listdir(img_dir):
                    if img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
                        lbl_file = img_file.rsplit('.', 1)[0] + '.txt'
                        lbl_path = os.path.join(lbl_dir, lbl_file)
                        if os.path.exists(lbl_path):
                            # Check label is not empty
                            if os.path.getsize(lbl_path) > 0:
                                all_pairs.append((
                                    os.path.join(img_dir, img_file),
                                    lbl_path,
                                    ds_cfg['class_map']  # class remapping for this dataset
                                ))
    
    # ---- Collect from auto-labeled datasets ----
    for name, info in auto_labeled_dirs.items():
        img_dir = info['images']
        lbl_dir = info['labels']
        if os.path.exists(img_dir) and os.path.exists(lbl_dir):
            for img_file in os.listdir(img_dir):
                if img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    lbl_file = img_file.rsplit('.', 1)[0] + '.txt'
                    lbl_path = os.path.join(lbl_dir, lbl_file)
                    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
                        all_pairs.append((
                            os.path.join(img_dir, img_file),
                            lbl_path,
                            {0: 0}  # auto-labeled: person→fall
                        ))
    
    # ---- Shuffle and split ----
    random.shuffle(all_pairs)
    n_total = len(all_pairs)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    
    train_pairs = all_pairs[:n_train]
    val_pairs = all_pairs[n_train:n_train + n_val]
    test_pairs = all_pairs[n_train + n_val:]
    
    # ---- Copy files ----
    def copy_pairs(pairs, split_name):
        for i, (img_path, lbl_path, class_map) in enumerate(pairs):
            img_ext = Path(img_path).suffix
            dst_img = f"{output_dir}/{split_name}/images/{split_name}_{i:06d}{img_ext}"
            dst_lbl = f"{output_dir}/{split_name}/labels/{split_name}_{i:06d}.txt"
            shutil.copy2(img_path, dst_img)
            remap_label_file(lbl_path, dst_lbl, class_map)
    
    copy_pairs(train_pairs, 'train')
    copy_pairs(val_pairs, 'val')
    copy_pairs(test_pairs, 'test')
    
    print(f"Unified dataset created at: {output_dir}")
    print(f"  Train: {len(train_pairs)} images")
    print(f"  Val:   {len(val_pairs)} images")
    print(f"  Test:  {len(test_pairs)} images")
    print(f"  Total: {n_total} images")
    
    return output_dir

# Collect Roboflow dataset paths with class mapping info
# Dataset 1 (humna): class 0='person' → remap 0→0 (person IS fall in this dataset)
# Dataset 2 (yolo-xvnzo): class 0='fall', 1='not-fallen' → keep as-is
roboflow_datasets_config = [
    {'path': f'{DATASET_DIR}/falling_pose_1', 'class_map': {0: 0}},  # person → fall
    {'path': f'{DATASET_DIR}/falling_pose_2', 'class_map': {0: 0, 1: 1}},  # fall→fall, not-fallen→not-fallen
]
roboflow_paths = [cfg['path'] for cfg in roboflow_datasets_config if os.path.exists(cfg['path'])]

# Merge everything
unified_path = merge_datasets(roboflow_datasets_config, labeled_dirs, UNIFIED_DIR)

print("\n✓ Step 5 complete: Datasets merged")

Unified dataset created at: /kaggle/working/datasets/ardumedics_unified_pose
  Train: 818 images
  Val:   153 images
  Test:  52 images
  Total: 1023 images

✓ Step 5 complete: Datasets merged


In [7]:
# ============================================================
# STEP 5b: Free disk space by removing intermediate files
# Kaggle /kaggle/working/ has a 20GB limit!
# ============================================================

print('Disk usage BEFORE cleanup:')
!df -h /kaggle/working 2>/dev/null || echo '(df not available)'

# Delete intermediate extracted frames and their auto-labels
# These are now safely merged into the unified dataset
cleanup_dirs = [
    'ur_fall_frames', 'ur_fall_frames_labels',
    'le2i_frames', 'le2i_frames_labels',
    'multi_cam_frames', 'multi_cam_frames_labels',
    'fall_video_frames', 'fall_video_frames_labels',
    'fall_detection_images', 'fall_detection_images_labels',
    'falling_pose_1',  # Original Roboflow copy (merged into unified)
    'falling_pose_2',  # Original Roboflow copy (merged into unified)
]

freed_mb = 0
for dir_name in cleanup_dirs:
    dir_path = os.path.join(DATASET_DIR, dir_name)
    if os.path.exists(dir_path):
        # Calculate size before deleting
        total_size = 0
        for root, dirs, files in os.walk(dir_path):
            for f in files:
                fp = os.path.join(root, f)
                if os.path.exists(fp):
                    total_size += os.path.getsize(fp)
        freed_mb += total_size / (1024 * 1024)
        shutil.rmtree(dir_path, ignore_errors=True)
        print(f'  Deleted {dir_name}/ ({total_size/(1024*1024):.1f} MB)')

print(f'\nTotal freed: {freed_mb:.1f} MB')
print('\nDisk usage AFTER cleanup:')
!df -h /kaggle/working 2>/dev/null || echo '(df not available)'
print('\n✓ Step 5b complete: Disk space freed for training')

Disk usage BEFORE cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  118M   20G   1% /kaggle/working
  Deleted falling_pose_1/ (36.4 MB)
  Deleted falling_pose_2/ (15.2 MB)

Total freed: 51.7 MB

Disk usage AFTER cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   61M   20G   1% /kaggle/working

✓ Step 5b complete: Disk space freed for training


---
## Step 5b: Free Disk Space (Kaggle 20GB Limit)

Kaggle limits `/kaggle/working/` to **20GB**. We just duplicated all data
into the unified dataset. Now delete the intermediate extracted frames
and their auto-labels to free up disk space before training starts.

**This is critical** — without cleanup, the notebook will crash with
`OSError: [Errno 28] No space left on device` during training.

---
## Step 6: Create data.yaml and Verify Dataset

Create the YOLOv8 data configuration file and verify that all labels are valid.

In [8]:
# ============================================================
# STEP 6: Create data.yaml and verify dataset integrity
# Notebook: 01 | Step: 6 of 10
# After this: Train YOLOv8n-Pose model
# ============================================================

# Verify label format (YOLOv8-Pose expects: class cx cy w h kx1 ky1 v1 kx2 ky2 v2 ...)
def verify_labels(dataset_dir, split='train', max_check=100):
    """Verify label files have correct format for YOLOv8-Pose."""
    label_dir = os.path.join(dataset_dir, split, 'labels')
    if not os.path.exists(label_dir):
        print(f"  No {split} labels directory found!")
        return False
    
    label_files = glob.glob(os.path.join(label_dir, '*.txt'))
    errors = 0
    
    for lf in label_files[:max_check]:
        with open(lf, 'r') as f:
            for line_idx, line in enumerate(f):
                parts = line.strip().split()
                # YOLOv8-Pose format: class + 4 bbox + 17*3 keypoints = 56 values
                # Or could be fewer keypoints
                if len(parts) < 5:
                    errors += 1
                    continue
                
                # First value should be class (integer)
                try:
                    cls = int(parts[0])
                except ValueError:
                    errors += 1
                    continue
    
    total = min(len(label_files), max_check)
    print(f"  {split}: Checked {total} label files, {errors} errors")
    return errors == 0

# Verify all splits
for split in ['train', 'val', 'test']:
    verify_labels(UNIFIED_DIR, split)

# Create data.yaml
data_yaml = {
    'path': UNIFIED_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {0: 'fall', 1: 'not-fallen'},
    'kpt_shape': [17, 3],  # 17 COCO keypoints, each with (x, y, visibility)
    'flip_idx': [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
}

yaml_path = f"{DATASET_DIR}/ardumedics_pose.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"\ndata.yaml created at: {yaml_path}")
print(f"  Dataset path: {UNIFIED_DIR}")
print(f"  Keypoint shape: 17 keypoints x 3 (x, y, visibility)")

# Count total images
total_images = 0
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(UNIFIED_DIR, split, 'images')
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        total_images += n
        print(f"  {split}: {n} images")
print(f"  TOTAL: {total_images} images")

print("\n✓ Step 6 complete: Dataset verified and data.yaml created")

  train: Checked 100 label files, 0 errors
  val: Checked 100 label files, 0 errors
  test: Checked 52 label files, 0 errors

data.yaml created at: /kaggle/working/datasets/ardumedics_pose.yaml
  Dataset path: /kaggle/working/datasets/ardumedics_unified_pose
  Keypoint shape: 17 keypoints x 3 (x, y, visibility)
  train: 818 images
  val: 153 images
  test: 52 images
  TOTAL: 1023 images

✓ Step 6 complete: Dataset verified and data.yaml created


---
## Step 7: Train YOLOv8n-Pose Model

This is the main training step. We use:
- **AdamW optimizer** (better convergence than SGD for fine-tuning)
- **Cosine annealing LR** (smooth learning rate decay)
- **150 epochs** with patience=40 early stopping
- **Mosaic + MixUp** augmentation for robustness
- **imgsz=640** for good accuracy/speed balance

The nano model will be our **primary deployment model** on Raspberry Pi 5.

In [9]:
# ============================================================
# STEP 7: Train YOLOv8n-Pose model
# Notebook: 01 | Step: 7 of 10
# After this: Evaluate model performance on validation set
# ============================================================

# Initialize model from pre-trained COCO weights
model = YOLO('yolov8n-pose.pt')

# Training configuration - OPTIMIZED for SOTA results
TRAIN_CONFIG = {
    'data': yaml_path,
    'epochs': 150,          # Extended training (was 100, increased for 12hr budget)
    'imgsz': 640,           # Standard YOLO resolution
    'batch': 32,  # Doubled for T4x2 (was 16 for single T4)            # T4 GPU can handle batch 16 for nano model
    'patience': 40,         # Early stopping patience (generous)
    'device': [0, 1],  # T4x2 Dual GPU (DDP)            # GPU
    'seed': SEED,
    'workers': 4,  # Kaggle has 4 CPU cores; 8 causes DataLoader crashes with dual GPU           # Kaggle T4 has 4 CPU cores, 8 workers is fine
    'optimizer': 'AdamW',   # Better than SGD for fine-tuning
    'lr0': 0.001,           # Initial learning rate
    'lrf': 0.01,            # Final LR factor (lr0 * lrf)
    'cos_lr': True,         # Cosine annealing schedule
    'warmup_epochs': 5,     # Gradual warmup
    'warmup_bias_lr': 0.01, # Warmup bias learning rate
    
    # Augmentation settings
    'augment': True,
    'mosaic': 1.0,          # Mosaic augmentation probability
    'mixup': 0.1,           # MixUp augmentation probability
    'copy_paste': 0.3,      # Copy-paste augmentation (good for pose)
    'degrees': 15.0,        # Rotation augmentation range
    'translate': 0.2,       # Translation augmentation
    'scale': 0.5,           # Scale augmentation
    'fliplr': 0.5,          # Horizontal flip probability
    'hsv_h': 0.015,         # HSV hue augmentation
    'hsv_s': 0.7,           # HSV saturation augmentation
    'hsv_v': 0.4,           # HSV value augmentation
    
    # Training settings
    'project': '/kaggle/working/runs',
    'name': 'ardumedics_nano_pose',
    'exist_ok': True,
    'pretrained': True,
    'verbose': True,
    
    # Logging
    'val': True,            # Validate every epoch
    'plots': True,          # Generate training plots
    'save': True,           # Save checkpoints
    'save_period': 25,      # Save checkpoint every 25 epochs
}

print("Starting YOLOv8n-Pose training...")
print(f"  Model: yolov8n-pose (Nano)")
print(f"  Epochs: {TRAIN_CONFIG['epochs']}")
print(f"  Optimizer: {TRAIN_CONFIG['optimizer']}")
print(f"  Learning rate: {TRAIN_CONFIG['lr0']} (cosine annealing)")
print(f"  Batch size: {TRAIN_CONFIG['batch']}")
print(f"  Image size: {TRAIN_CONFIG['imgsz']}")
print()

# RUN TRAINING
results = model.train(**TRAIN_CONFIG)

print("\n✓ Step 7 complete: Training finished")

Starting YOLOv8n-Pose training...
  Model: yolov8n-pose (Nano)
  Epochs: 150
  Optimizer: AdamW
  Learning rate: 0.001 (cosine annealing)
  Batch size: 32
  Image size: 640

Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/datasets/ardumedics_pose.yaml, degrees=15.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=Fal

---
## Step 8: Evaluate Model Performance

Evaluate the trained model on validation and test sets. Record all metrics
for comparison with other model sizes (Notebooks 02 and 03).

In [10]:
# ============================================================
# STEP 8: Evaluate model performance
# Notebook: 01 | Step: 8 of 10
# After this: Export model to edge deployment formats
# ============================================================

# Load best model from training
best_model_path = '/kaggle/working/runs/ardumedics_nano_pose/weights/best.pt'
best_model = YOLO(best_model_path)

# ---- Validation Set Evaluation ----
print("=" * 60)
print("VALIDATION SET EVALUATION")
print("=" * 60)
val_results = best_model.val(data=yaml_path, split='val', verbose=True)

print(f"\nBox Metrics (Detection):")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall:    {val_results.box.mr:.4f}")
print(f"  mAP@50:    {val_results.box.map50:.4f}")
print(f"  mAP@50-95: {val_results.box.map:.4f}")

print(f"\nPose Metrics (Keypoints):")
print(f"  mAP@50 (Pose):    {val_results.pose.map50:.4f}")
print(f"  mAP@50-95 (Pose): {val_results.pose.map:.4f}")

# ---- Test Set Evaluation ----
print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)
test_results = best_model.val(data=yaml_path, split='test', verbose=True)

print(f"\nBox Metrics (Detection):")
print(f"  Precision: {test_results.box.mp:.4f}")
print(f"  Recall:    {test_results.box.mr:.4f}")
print(f"  mAP@50:    {test_results.box.map50:.4f}")
print(f"  mAP@50-95: {test_results.box.map:.4f}")

print(f"\nPose Metrics (Keypoints):")
print(f"  mAP@50 (Pose):    {test_results.pose.map50:.4f}")
print(f"  mAP@50-95 (Pose): {test_results.pose.map:.4f}")

# ---- Save metrics to JSON for later comparison ----
metrics_01 = {
    'notebook': '01_YOLOv8n_Pose',
    'model': 'yolov8n-pose',
    'model_size': 'n',
    'best_mAP50': float(val_results.box.map50),
    'timestamp': datetime.now().isoformat(),
    'val': {
        'box_precision': float(val_results.box.mp),
        'box_recall': float(val_results.box.mr),
        'box_map50': float(val_results.box.map50),
        'box_map': float(val_results.box.map),
        'pose_map50': float(val_results.pose.map50),
        'pose_map': float(val_results.pose.map),
    },
    'test': {
        'box_precision': float(test_results.box.mp),
        'box_recall': float(test_results.box.mr),
        'box_map50': float(test_results.box.map50),
        'box_map': float(test_results.box.map),
        'pose_map50': float(test_results.pose.map50),
        'pose_map': float(test_results.pose.map),
    }
}

metrics_path = '/kaggle/working/metrics_notebook01_nano.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_01, f, indent=2)

print(f"\nMetrics saved to: {metrics_path}")
print("\n✓ Step 8 complete: Evaluation finished")

VALIDATION SET EVALUATION
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-pose summary (fused): 82 layers, 3,290,159 parameters, 0 gradients, 9.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1764.0±814.4 MB/s, size: 94.9 KB)
val: Scanning /kaggle/working/datasets/ardumedics_unified_pose/val/labels.cache... 153 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 153/153 42.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 4.1it/s 2.4s
                   all        153        159      0.637      0.841      0.712      0.466      0.341      0.391       0.25     0.0753
                  fall        121        121      0.686      0.893      0.885      0.577      0.462      0.545      0.416      0.139
            not-fallen         35         38      0.588      0.789      0.539      0.355      0.219      0.237     0.0846

---
## Step 9: Export Model to Edge Deployment Formats

Export the trained model to formats optimized for Raspberry Pi 5 deployment:
- **NCNN**: Best performance on ARM CPUs (primary format for Pi 5)
- **TFLite**: With INT8 quantization (secondary format)
- **ONNX**: Universal format (fallback + compatibility)

In [11]:
# ============================================================
# STEP 9: Export model to edge deployment formats
# Notebook: 01 | Step: 9 of 10
# After this: Generate Temporal Pose Consistency tracker code
# ============================================================

EXPORT_DIR = '/kaggle/working/exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

# ---- Export 1: NCNN (Best for ARM/Raspberry Pi 5) ----
print("Exporting to NCNN format...")
try:
    ncnn_path = best_model.export(format='ncnn', imgsz=640, simplify=True)
    print(f"  NCNN exported to: {ncnn_path}")
except Exception as e:
    print(f"  NCNN export failed: {e}")
    ncnn_path = None

# ---- Export 2: TFLite with INT8 quantization ----
print("\nExporting to TFLite (INT8) format...")
try:
    tflite_path = best_model.export(format='tflite', imgsz=640, int8=True)
    print(f"  TFLite exported to: {tflite_path}")
except Exception as e:
    print(f"  TFLite export failed: {e}")
    tflite_path = None

# ---- Export 3: ONNX ----
print("\nExporting to ONNX format...")
try:
    onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True, dynamic=True)
    print(f"  ONNX exported to: {onnx_path}")
except Exception as e:
    print(f"  ONNX export failed: {e}")
    onnx_path = None

# ---- List exported model sizes ----
print("\n" + "=" * 60)
print("EXPORTED MODEL SIZES")
print("=" * 60)
for fmt_name, fmt_path in [('PyTorch (best.pt)', best_model_path),
                            ('NCNN', ncnn_path),
                            ('TFLite (INT8)', tflite_path),
                            ('ONNX', onnx_path)]:
    if fmt_path and os.path.exists(str(fmt_path)):
        size_mb = os.path.getsize(str(fmt_path)) / (1024 * 1024)
        print(f"  {fmt_name:25s}: {size_mb:.2f} MB")
    elif fmt_path is None:
        print(f"  {fmt_name:25s}: EXPORT FAILED")
    else:
        print(f"  {fmt_name:25s}: FILE NOT FOUND")

# Update metrics JSON with export info
metrics_01['exports'] = {
    'ncnn_path': str(ncnn_path) if ncnn_path else 'FAILED',
    'tflite_path': str(tflite_path) if tflite_path else 'FAILED',
    'onnx_path': str(onnx_path) if onnx_path else 'FAILED',
    'pytorch_size_mb': os.path.getsize(best_model_path) / (1024 * 1024),
}
with open(metrics_path, 'w') as f:
    json.dump(metrics_01, f, indent=2)

print("\n✓ Step 9 complete: Models exported")

Exporting to NCNN format...
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ NCNN export does not support end2end models, disabling end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/kaggle/working/runs/ardumedics_nano_pose/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 57, 8400) (6.5 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 193ms
 Downloaded ncnn
Prepared 1 package in 218ms
Installed 1 package in 7ms
 + ncnn==1.0.20260114

requirements: AutoUpdate success ✅ 1.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pnnx'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /u

pnnxparam = /kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Bottleneck
inline module = ultralytics.nn.modules.block.C2f
inline mo

NCNN: export success ✅ 6.4s, saved as '/kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model' (12.8 MB)

Export complete (6.7s)
Results saved to /kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model
Predict:         yolo predict task=pose model=/kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=pose model=/kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model imgsz=640 data=/kaggle/working/datasets/ardumedics_pose.yaml  
Visualize:       https://netron.app
  NCNN exported to: /kaggle/working/runs/ardumedics_nano_pose/weights/best_ncnn_model

Exporting to TFLite (INT8) format...
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8-pose.yaml'.

PyTorch: starting from '/kaggle/working/runs/ardumedics_nano_pose/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and ou

E0000 00:00:1779210459.048311      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779210459.164467      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779210460.172195      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779210460.172220      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779210460.172222      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779210460.172225      23 computation_placer.cc:177] computation placer already registered. Please check linka

requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx2tf>=1.26.3,<1.29.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 3.71s
 Downloaded ai-edge-litert
Prepared 5 packages in 267ms
Installed 5 packages in 7ms
 + ai-edge-litert==2.1.5
 + backports-strenum==1.3.1
 + onnx-graphsurgeon==0.6.1
 + onnx2tf==1.28.8
 + sng4onnx==2.0.1

requirements: AutoUpdate success ✅ 4.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.0...
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /kaggle/working/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 52.7files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...


I0000 00:00:1779210497.864883      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12827 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779210497.870110      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1779210501.922787      23 cuda_dnn.cc:529] Loaded cuDNN version 91002


Saved artifact at '/kaggle/working/runs/ardumedics_nano_pose/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 57, 8400), dtype=tf.float32, name=None)
Captures:
  136591291771856: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  136591291770320: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  136591291771088: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
  136591291774736: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  136591291776272: TensorSpec(shape=(3, 3, 16, 32), dtype=tf.float32, name=None)
  136591291773392: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  136591291776464: TensorSpec(shape=(1, 1, 32, 32), dtype=tf.float32, name=None)
  136591291772048: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  136591291776848: TensorSpec(shape=(4,), dtype=tf.int64, n

I0000 00:00:1779210508.071060      23 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1779210508.071255      23 single_machine.cc:374] Starting new session
I0000 00:00:1779210508.084546      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12827 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779210508.086006      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
W0000 00:00:1779210508.899596      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779210508.899634      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1779210509.726833      23 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1779210509.727

W0000 00:00:1779210513.815819      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779210513.815851      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1779210513.849291      23 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1779210519.947700      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779210519.947750      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1779210525.916516      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779210525.916609      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
W0000 00:00:1779210535.470100      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_for

TensorFlow SavedModel: export success ✅ 113.9s, saved as '/kaggle/working/runs/ardumedics_nano_pose/weights/best_saved_model' (43.1 MB)

TensorFlow Lite: starting export with tensorflow 2.19.0...
TensorFlow Lite: export success ✅ 0.0s, saved as '/kaggle/working/runs/ardumedics_nano_pose/weights/best_saved_model/best_int8.tflite' (3.6 MB)

Export complete (114.2s)
Results saved to /kaggle/working/runs/ardumedics_nano_pose/weights/best_saved_model/best_int8.tflite
Predict:         yolo predict task=pose model=/kaggle/working/runs/ardumedics_nano_pose/weights/best_saved_model/best_int8.tflite imgsz=640 int8
Validate:        yolo val task=pose model=/kaggle/working/runs/ardumedics_nano_pose/weights/best_saved_model/best_int8.tflite imgsz=640 data=/kaggle/working/datasets/ardumedics_pose.yaml int8 
Visualize:       https://netron.app
  TFLite exported to: /kaggle/working/runs/ardumedics_nano_pose/weights/best_saved_model/best_int8.tflite

Exporting to ONNX format...
Ultralytics 8.4.51 🚀 Pyt

---
## Step 10: Temporal Pose Consistency (TPC) Tracker

This is our **novel contribution** for the Q1 paper. The TPC tracker monitors
keypoint changes across consecutive frames to distinguish real falls from
benign actions (sitting, lying down, bending).

This code will be used on Raspberry Pi 5 for real-time inference.

In [12]:
# ============================================================
# STEP 10: Temporal Pose Consistency (TPC) Tracker
# Notebook: 01 | Step: 10 of 10
# This is a NOVEL CONTRIBUTION for the ArduMedics Q1 paper
# After this: Save all outputs and summary
# ============================================================

tpc_code = '''"""
ArduMedics Temporal Pose Consistency (TPC) Tracker
===================================================
Novel contribution: Tracks pose changes over time to distinguish
real falls from benign actions (sitting, lying, bending).

This code runs on Raspberry Pi 5 alongside the YOLOv8-Pose model.

Key Innovation:
- Velocity-based fall detection (not just static pose)
- Multi-frame confirmation (reduces false positives)
- Recovery detection (distinguishes fall from lying down)
- Configurable sensitivity for different environments

Author: ArduMedics AI Team
Notebook: 01 | Step: 10
"""

import numpy as np
from collections import deque
from dataclasses import dataclass
from typing import Optional, List, Tuple


@dataclass
class PoseFrame:
    """Single frame of pose data with timestamp."""
    keypoints: np.ndarray  # Shape: (17, 3) - x, y, confidence
    timestamp: float
    bbox: Optional[np.ndarray] = None  # Bounding box [x1, y1, x2, y2]


class TemporalPoseTracker:
    """
    Temporal Pose Consistency (TPC) Tracker for Fall Detection.
    
    Monitors pose changes across consecutive frames to detect falls
    while filtering out benign actions like sitting or bending.
    
    Args:
        window_size: Number of frames to keep in history buffer
        fall_velocity_threshold: Vertical velocity threshold for fall (pixels/sec)
        horizontal_ratio_threshold: Ratio below which torso is considered horizontal
        ground_proximity_threshold: Pixel distance threshold for near-ground
        confirmation_frames: Number of consecutive frames needed to confirm fall
        recovery_timeout: Seconds after which a lying person is considered recovered
    """
    
    def __init__(
        self,
        window_size: int = 30,
        fall_velocity_threshold: float = 150.0,
        horizontal_ratio_threshold: float = 0.6,
        ground_proximity_threshold: float = 80.0,
        confirmation_frames: int = 5,
        recovery_timeout: float = 10.0,
    ):
        self.window_size = window_size
        self.fall_velocity_threshold = fall_velocity_threshold
        self.horizontal_ratio_threshold = horizontal_ratio_threshold
        self.ground_proximity_threshold = ground_proximity_threshold
        self.confirmation_frames = confirmation_frames
        self.recovery_timeout = recovery_timeout
        
        # Frame history buffer
        self.frame_buffer: deque = deque(maxlen=window_size)
        
        # Fall detection state
        self.fall_counter: int = 0
        self.fall_detected: bool = False
        self.fall_timestamp: Optional[float] = None
        self.recovery_detected: bool = False
    
    def _get_keypoint(self, keypoints: np.ndarray, idx: int) -> Tuple[float, float, float]:
        """Extract (x, y, confidence) for a keypoint index."""
        if idx < len(keypoints):
            return keypoints[idx][0], keypoints[idx][1], keypoints[idx][2]
        return 0.0, 0.0, 0.0
    
    def compute_torso_metrics(self, keypoints: np.ndarray) -> dict:
        """
        Compute torso orientation and position metrics from keypoints.
        
        COCO Keypoint indices:
            5: Left Shoulder   6: Right Shoulder
            11: Left Hip       12: Right Hip
            15: Left Ankle     16: Right Ankle
        
        Returns dict with:
            - shoulder_y, hip_y, ankle_y: Average Y coordinates
            - torso_height: Vertical distance shoulder-to-hip
            - torso_width: Horizontal distance shoulder-center to hip-center
            - is_horizontal: Whether torso is more horizontal than vertical
            - near_ground: Whether person is close to ground level
            - aspect_ratio: torso_height / torso_width ratio
        """
        ls = self._get_keypoint(keypoints, 5)
        rs = self._get_keypoint(keypoints, 6)
        lh = self._get_keypoint(keypoints, 11)
        rh = self._get_keypoint(keypoints, 12)
        la = self._get_keypoint(keypoints, 15)
        ra = self._get_keypoint(keypoints, 16)
        
        # Average Y coordinates (higher Y = lower in image)
        shoulder_y = (ls[1] + rs[1]) / 2
        hip_y = (lh[1] + rh[1]) / 2
        ankle_y = (la[1] + ra[1]) / 2
        
        # Torso dimensions
        torso_height = abs(shoulder_y - hip_y)
        shoulder_cx = (ls[0] + rs[0]) / 2
        hip_cx = (lh[0] + rh[0]) / 2
        torso_width = abs(shoulder_cx - hip_cx)
        
        # Avoid division by zero
        if torso_width < 1e-6:
            torso_width = 1e-6
        
        aspect_ratio = torso_height / torso_width
        is_horizontal = aspect_ratio < self.horizontal_ratio_threshold
        near_ground = abs(shoulder_y - ankle_y) < self.ground_proximity_threshold
        
        return {
            'shoulder_y': shoulder_y,
            'hip_y': hip_y,
            'ankle_y': ankle_y,
            'torso_height': torso_height,
            'torso_width': torso_width,
            'aspect_ratio': aspect_ratio,
            'is_horizontal': is_horizontal,
            'near_ground': near_ground,
        }
    
    def compute_vertical_velocity(self) -> float:
        """
        Compute vertical velocity of the shoulder center over recent frames.
        Positive velocity = moving DOWN (toward fall direction).
        
        Returns:
            Vertical velocity in pixels per second
        """
        if len(self.frame_buffer) < 2:
            return 0.0
        
        recent = list(self.frame_buffer)
        
        # Use last 5 frames for velocity estimation
        n_vel = min(5, len(recent))
        vel_frames = recent[-n_vel:]
        
        # Compute shoulder Y changes
        velocities = []
        for i in range(1, len(vel_frames)):
            prev_metrics = self.compute_torso_metrics(vel_frames[i-1].keypoints)
            curr_metrics = self.compute_torso_metrics(vel_frames[i].keypoints)
            
            dt = vel_frames[i].timestamp - vel_frames[i-1].timestamp
            if dt > 0:
                dy = curr_metrics['shoulder_y'] - prev_metrics['shoulder_y']
                velocities.append(dy / dt)
        
        return np.mean(velocities) if velocities else 0.0
    
    def update(self, keypoints: np.ndarray, timestamp: float, bbox: np.ndarray = None) -> dict:
        """
        Process a new frame and return fall detection status.
        
        Args:
            keypoints: Array of shape (17, 3) with (x, y, confidence)
            timestamp: Frame timestamp in seconds
            bbox: Optional bounding box [x1, y1, x2, y2]
        
        Returns:
            dict with:
                - fall_detected: bool - Whether a fall is currently detected
                - fall_confidence: float - Confidence of fall detection [0, 1]
                - is_horizontal: bool - Whether torso is horizontal
                - near_ground: bool - Whether person is near ground
                - velocity: float - Vertical velocity
                - state: str - Current state ('normal', 'falling', 'fallen', 'recovered')
        """
        # Add frame to buffer
        frame = PoseFrame(keypoints=keypoints, timestamp=timestamp, bbox=bbox)
        self.frame_buffer.append(frame)
        
        # Compute current metrics
        metrics = self.compute_torso_metrics(keypoints)
        velocity = self.compute_vertical_velocity()
        
        # Fall detection logic
        # Condition 1: Torso is horizontal (not vertical/standing)
        # Condition 2: Person is near ground level
        # Condition 3: High downward velocity (rapid descent)
        
        is_falling_pose = metrics['is_horizontal'] and metrics['near_ground']
        is_rapid_descent = velocity > self.fall_velocity_threshold
        
        # Multi-frame confirmation
        if is_falling_pose or is_rapid_descent:
            self.fall_counter += 1
        else:
            self.fall_counter = max(0, self.fall_counter - 2)  # Decay faster than accumulation
        
        # Determine state
        if self.fall_counter >= self.confirmation_frames and not self.fall_detected:
            # NEW FALL DETECTED
            self.fall_detected = True
            self.fall_timestamp = timestamp
            state = 'fallen'
        elif self.fall_detected:
            # Check for recovery
            if not metrics['is_horizontal'] and not metrics['near_ground']:
                self.recovery_detected = True
                state = 'recovered'
            elif timestamp - self.fall_timestamp > self.recovery_timeout:
                # Timeout - person hasn't recovered
                state = 'fallen'
            else:
                state = 'fallen'
        elif is_rapid_descent:
            state = 'falling'
        else:
            state = 'normal'
            # Reset fall state if normal for a while
            if self.fall_counter == 0:
                self.fall_detected = False
                self.fall_timestamp = None
                self.recovery_detected = False
        
        # Compute fall confidence
        confidence = min(1.0, self.fall_counter / self.confirmation_frames)
        
        return {
            'fall_detected': self.fall_detected,
            'fall_confidence': confidence,
            'is_horizontal': metrics['is_horizontal'],
            'near_ground': metrics['near_ground'],
            'velocity': velocity,
            'aspect_ratio': metrics['aspect_ratio'],
            'state': state,
        }
    
    def reset(self):
        """Reset tracker state."""
        self.frame_buffer.clear()
        self.fall_counter = 0
        self.fall_detected = False
        self.fall_timestamp = None
        self.recovery_detected = False
'''

# Save TPC code
tpc_path = '/kaggle/working/temporal_pose_tracker.py'
with open(tpc_path, 'w') as f:
    f.write(tpc_code)

print(f"TPC Tracker code saved to: {tpc_path}")
print(f"Code length: {len(tpc_code)} characters")

# Quick test of TPC logic
exec(tpc_code)
tracker = TemporalPoseTracker()

# Simulate a fall: Standing -> Falling -> On ground
standing_kpts = np.zeros((17, 3))
standing_kpts[5] = [320, 100, 0.9]   # Left shoulder
standing_kpts[6] = [340, 100, 0.9]   # Right shoulder
standing_kpts[11] = [310, 250, 0.9]  # Left hip
standing_kpts[12] = [350, 250, 0.9]  # Right hip
standing_kpts[15] = [305, 450, 0.9]  # Left ankle
standing_kpts[16] = [355, 450, 0.9]  # Right ankle

fallen_kpts = np.zeros((17, 3))
fallen_kpts[5] = [100, 400, 0.9]   # Left shoulder (low, left)
fallen_kpts[6] = [100, 380, 0.9]   # Right shoulder (low, left)
fallen_kpts[11] = [250, 380, 0.9]  # Left hip (right of shoulder)
fallen_kpts[12] = [250, 400, 0.9]  # Right hip (right of shoulder)
fallen_kpts[15] = [400, 400, 0.9]  # Left ankle (far right)
fallen_kpts[16] = [400, 380, 0.9]  # Right ankle (far right)

print("\n--- TPC Tracker Test ---")
for i in range(10):
    result = tracker.update(standing_kpts, timestamp=i*0.1)
print(f"Standing: state={result['state']}, fall={result['fall_detected']}")

tracker.reset()
for i in range(15):
    result = tracker.update(fallen_kpts, timestamp=i*0.1)
print(f"Fallen:  state={result['state']}, fall={result['fall_detected']}, confidence={result['fall_confidence']:.2f}")

print("\n✓ Step 10 complete: TPC Tracker created and tested")

TPC Tracker code saved to: /kaggle/working/temporal_pose_tracker.py
Code length: 9688 characters

--- TPC Tracker Test ---
Standing: state=normal, fall=False
Fallen:  state=fallen, fall=True, confidence=1.00

✓ Step 10 complete: TPC Tracker created and tested


---
## Step 11: Save Outputs for Next Notebooks

Pack all outputs (trained model, metrics JSON, exports) into a directory
that can be saved as a **Kaggle Dataset** for use by Notebooks 05 and 06.

**Instructions**: After this notebook completes, go to the Output section
and create a new Dataset from the output. Name it something like
`ardumedics-nb01-nano-outputs`. Then add this dataset as input to NB05/NB06.

In [13]:
# ============================================================
# STEP 11: Save outputs for next notebooks
# Pack everything into /kaggle/working/nb01_outputs/ for Kaggle Dataset
# ============================================================

OUTPUT_PACK_DIR = '/kaggle/working/nb01_outputs'
os.makedirs(OUTPUT_PACK_DIR, exist_ok=True)

# Copy best model weights
import shutil
best_pt_src = '/kaggle/working/runs/ardumedics_nano_pose/weights/best.pt'
if os.path.exists(best_pt_src):
    shutil.copy2(best_pt_src, f'{OUTPUT_PACK_DIR}/best_nano.pt')
    print(f'  Copied best_nano.pt ({os.path.getsize(f"{OUTPUT_PACK_DIR}/best_nano.pt")/1e6:.1f} MB)')

# Copy metrics JSON
metrics_src = '/kaggle/working/metrics_notebook01_nano.json'
if os.path.exists(metrics_src):
    shutil.copy2(metrics_src, f'{OUTPUT_PACK_DIR}/metrics_notebook01_nano.json')
    print('  Copied metrics_notebook01_nano.json')

# Copy TPC tracker code
tpc_src = '/kaggle/working/temporal_pose_tracker.py'
if os.path.exists(tpc_src):
    shutil.copy2(tpc_src, f'{OUTPUT_PACK_DIR}/temporal_pose_tracker.py')
    print('  Copied temporal_pose_tracker.py')

# Copy exported models (if any)
export_dir = '/kaggle/working/exports'
if os.path.exists(export_dir):
    for item in os.listdir(export_dir):
        src = os.path.join(export_dir, item)
        dst = os.path.join(OUTPUT_PACK_DIR, item)
        if os.path.isdir(src):
            if not os.path.exists(dst):
                shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
    print(f'  Copied export files')

print(f'\nAll outputs packed to: {OUTPUT_PACK_DIR}')
print(f'Total size: {sum(os.path.getsize(os.path.join(OUTPUT_PACK_DIR, f)) for f in os.listdir(OUTPUT_PACK_DIR) if os.path.isfile(os.path.join(OUTPUT_PACK_DIR, f)))/1e6:.1f} MB')
print('\n>>> IMPORTANT: Save this output as a Kaggle Dataset! <<<')
print('    1. Click "Save Version" to run the notebook')
print('    2. After completion, go to Output tab')
print('    3. Create a new Dataset from the nb01_outputs/ folder')
print('    4. Name it: ardumedics-nb01-nano-outputs')
print('    5. Add this dataset as input to NB05 and NB06')
print('\n✓ Step 11 complete: Outputs packed for next notebooks')

  Copied best_nano.pt (6.8 MB)
  Copied metrics_notebook01_nano.json
  Copied temporal_pose_tracker.py
  Copied export files

All outputs packed to: /kaggle/working/nb01_outputs
Total size: 6.9 MB

>>> IMPORTANT: Save this output as a Kaggle Dataset! <<<
    1. Click "Save Version" to run the notebook
    2. After completion, go to Output tab
    3. Create a new Dataset from the nb01_outputs/ folder
    4. Name it: ardumedics-nb01-nano-outputs
    5. Add this dataset as input to NB05 and NB06

✓ Step 11 complete: Outputs packed for next notebooks


---
## Final Summary: Notebook 01

**Outputs saved in `/kaggle/working/`:**
- `runs/ardumedics_nano_pose/weights/best.pt` — Best PyTorch weights
- `exports/` — NCNN, TFLite, ONNX exported models
- `metrics_notebook01_nano.json` — All evaluation metrics
- `temporal_pose_tracker.py` — TPC Tracker code (novel contribution)
- `nb01_outputs/` — Packed outputs for NB05/NB06

**Next Steps:**
- **Notebook 02**: Train YOLOv8s-Pose (small model - higher accuracy)
- **Notebook 03**: Train YOLOv8m-Pose (medium model - highest accuracy)
- **Notebook 05**: Compare all models + hyperparameter sweep
- **Notebook 06**: Ensemble + final paper results

**IMPORTANT: Save output as Kaggle Dataset** for NB05/NB06 to access!
